# MDA Text Preprocessing Pipeline

**Input:** Folder of MDA `.txt` files named like `Alphabet Inc._10-Q_2025-09-30T00_00_00_English_MDA.txt`

**Output:** Two DataFrames:
1. **Document-level** (`df_doc`) — one row per filing, for company-level sentiment & topic modelling
2. **Sentence-level** (`df_sent`) — one row per sentence, for joint sentiment × topic analysis

## Imports & Config

In [15]:
import re
import os
import pandas as pd
import nltk
from pathlib import Path
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize
from gensim.models.phrases import Phrases, ENGLISH_CONNECTOR_WORDS

nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

# ── Config ──────────────────────────────────────────────────────────
MDA_FOLDER       = "../MDA"          
FILE_PATTERN     = "*_MDA.txt"        
MIN_SENT_TOKENS  = 5                
BIGRAM_MIN_COUNT = 10                 # n-gram: pair must appear ≥ this many times
BIGRAM_THRESHOLD = 3.0                # n-gram: NPMI-like score threshold


## Stop Words & Regex Patterns
All patterns carried over from the NVIDIA notebook, with company-specific terms generalised.

In [16]:
# ── Stop words ──────────────────────────────────────────────────────
FINANCE_STOPWORDS = {
    "herein", "thereof", "thereto", "therein", "hereby", "pursuant",
    "accordance", "aforementioned", "foregoing", "whereas", "whereby",
    "hereafter", "hereinafter",
    "form", "quarterly", "annual", "report", "filing", "fiscal",
    "quarter", "period", "ended", "condensed", "consolidated",
    "statements", "notes", "refer", "also", "following", "described",
    "set", "forth", "part", "item",
    "incorporated", "reincorporated", "headquartered", "mean", "means",
    "including", "included", "related", "thereto",
    # company names added dynamically below
}

# Common corporate terms that appear across all companies
FINANCE_STOPWORDS |= {
    "company", "corporation", "subsidiaries", "inc", "ltd", "limited",
    "platforms", "manufacturing",
}

GEO_STOPWORDS = {
    "alabama", "alaska", "arizona", "arkansas", "california", "colorado",
    "connecticut", "delaware", "florida", "georgia", "hawaii", "idaho",
    "illinois", "indiana", "iowa", "kansas", "kentucky", "louisiana",
    "maine", "maryland", "massachusetts", "michigan", "minnesota",
    "mississippi", "missouri", "montana", "nebraska", "nevada",
    "new_hampshire", "new_jersey", "new_mexico", "new_york", "north_carolina",
    "north_dakota", "ohio", "oklahoma", "oregon", "pennsylvania",
    "rhode_island", "south_carolina", "south_dakota", "tennessee", "texas",
    "utah", "vermont", "virginia", "washington", "west_virginia",
    "wisconsin", "wyoming",
    "ca", "de", "tx", "ny", "wa", "nv", "fl", "ga", "il", "ma",
    "san_francisco", "san_jose", "santa_clara", "santa", "clara",
    "palo_alto", "menlo_park", "mountain_view", "sunnyvale", "cupertino",
    "redwood_city", "fremont", "san_diego", "los_angeles", "seattle",
    "portland", "austin", "boston", "new_york_city", "new_york",
    "chicago", "denver", "atlanta", "miami",
    "delaware", "nevada", "cayman_islands", "bermuda", "ireland",
    "luxembourg", "singapore", "hong_kong", "switzerland",
    "united_states", "united_kingdom", "china", "japan", "germany",
    "france", "india", "taiwan", "south_korea", "israel", "canada",
    "australia", "netherlands", "sweden", "finland",
    "north_america", "south_america", "latin_america", "europe",
    "asia_pacific", "middle_east", "apac", "emea", "americas",
}

STOPWORDS = set(stopwords.words("english")) | FINANCE_STOPWORDS | GEO_STOPWORDS

In [ ]:
# ── FLS boundary ───────────────────────────────────────────────────
FLS_START_ANCHORS = [
    r"Overview\s+Our\s+Company\s+and\s+Our\s+Businesses",
    r"Overview\s+Our\s+Company",
    r"(?:Overview|OVERVIEW)\s*\n",                # generic "Overview" heading
    r"Management.{0,20}s Discussion and Analysis", # standard MDA header
    r"Results of Operations",                      # fallback: skip straight to results
    r"Executive\s+Summary"
]

_FLS_PATTERN = re.compile("|".join(FLS_START_ANCHORS), re.IGNORECASE)

# ── Table removal: atomic building blocks ──────────────────────────
_VAL = (
    r'(?:\$\s*)?-?[\d,]+(?:\.\d+)?\s*(?:%|pts)?'
    r'|\([0-9,\.]+\)\s*(?:%|pts)?'
)
_DATE = (
    r'(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)'
    r'\s+\d{1,2},?\s*\d{4}'
)
_PERIOD = (
    r'Three\s+Months\s+Ended'
    r'|Six\s+Months\s+Ended'
    r'|Nine\s+Months\s+Ended'
    r'|Twelve\s+Months\s+Ended'
    r'|Quarter-over-Quarter\s+Change'
    r'|Year-over-Year\s+Change'
    r'|\$\s*Change'
    r'|%\s*Change'
)
_UNITS = r'\(\$?\s*[Ii]n\s+(?:millions|billions)[^)]*\)'

# ── Table removal: compiled patterns ───────────────────────────────
RE_HEADER_ROW = re.compile(
    r'(?:{date}|{period}|{units})'
    r'(?:\s+(?:{date}|{period}|{units}))*'
    .format(date=_DATE, period=_PERIOD, units=_UNITS),
    re.IGNORECASE
)
RE_DATA_ROW = re.compile(
    r'(?<![.!?]\s)'
    r'[A-Za-z][A-Za-z0-9 \&\(\),\-\.]{{1,50}}'
    r'(?:\s+(?:{val})){{2,}}'
    .format(val=_VAL),
    re.IGNORECASE
)
RE_FOLLOWING_TABLE = re.compile(r'[Tt]he following table[^.]+\.', re.IGNORECASE)
RE_ORPHAN_NUMS = re.compile(
    r'(?<!\w)(?:{val})(?:\s+(?:{val})){{0,3}}(?!\s*\w{{4}})'.format(val=_VAL),
    re.IGNORECASE
)
RE_PAGE_NUM = re.compile(r'\b\d{1,3}\b(?=\s+[A-Z])')

# ── Structural noise ───────────────────────────────────────────────
RE_ITEM_TAG   = re.compile(r'\bItem\s+\d+[A-Z]?\.\s*')
RE_PART_TAG   = re.compile(r'\bPart\s+[IVX]+,?\s*')
RE_COPYRIGHT  = re.compile(r'©?\s*\d{4}\s+\w[^.]*(?:Corporation|Inc|LLC|Ltd)[^.]*\.')
RE_WHITESPACE = re.compile(r'\s{2,}')

# ── N-gram blacklist ───────────────────────────────────────────────
NGRAM_BLACKLIST_PATTERNS = [
    r'^\s*[,;:\-\(\)]+',
    r'[,;:\-\(\)]+\s*$',
    r'^-\w+',
    r'\b(\w{4,})_\1\b',
    r'\b(\w{4,})_\w+_\1\b',
    r'(\b\w+_\w+)_\w*_?\1',
    r'\b(santa_clara|san_jose|palo_alto|menlo_park|santa_monica)\b',
    r'\b(california|delaware|nevada|washington|new_york|texas|georgia)\b',
    r'\b(san|santa|los|las|new|mount|fort|north|south|east|west)_\b',
    r'\bsan_(francisco|jose|diego|antonio|mateo|bernardino)\b',
    r'\bsanta_(clara|monica|barbara|ana|cruz|rosa)\b',
    r'\blos_(angeles|altos|gatos)\b',
    r'\bnew_(york|jersey|mexico|hampshire|orleans|haven)\b',
    r'\bfort_(worth|lauderdale|wayne|collins)\b',
    r'\bmount_(view|pleasant)\b',
    r'\b(california|delaware|nevada|washington|texas).*\b(california|delaware|nevada|washington|texas)\b',
    r'\bnote_financial\b',
    r'\b(form_10|10_k|10_q|8_k)\b',
    r'\b(exhibit|item_\d+|part_[ivx]+)\b',
    r'\b(one|two|three|four|five|reportable)_segment',
    r'\breporting_segment',
    r'\b(entered|continue|remained|could|would|should)_may\b',
    r'\bmay_(continue|remain|result|impact|affect|cause)\b',
    r'\b(first|second|third|fourth)_quarter_\w+_quarter\b',
    r'^\d+_',
]

## Helper Functions

In [ ]:
def parse_filename(filename: str) -> dict:
    """
    Parse: Alphabet Inc._10-Q_2025-09-30T00_00_00_English_MDA.txt
    Company name = everything before the first _10-K or _10-Q.
    Returns dict with: company_name, filing_type, filing_date, year, quarter
    """
    stem = Path(filename).stem
    m = re.match(
        r'^(?P<company>.+)_(?P<ftype>10-[KQ])_(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})(?:T[\d_]+)?_(?P<section>.+)',
        stem
    )
    if not m:
        return None

    company_name = m.group('company')
    filing_type  = m.group('ftype')
    year         = m.group('year')
    month        = int(m.group('month'))
    day          = m.group('day')
    quarter      = f"Q{(month - 1) // 3 + 1}"
    filing_date  = f"{year}-{m.group('month')}-{day}"

    return {
        "company_name": company_name,
        "filing_type":  filing_type,
        "filing_date":  filing_date,
        "year":         year,
        "quarter":      quarter,
    }

def is_blacklisted(ngram: str) -> bool:
    return any(re.search(pat, ngram, re.IGNORECASE) for pat in NGRAM_BLACKLIST_PATTERNS)


def strip_fls(raw: str) -> str:
    match = _FLS_PATTERN.search(raw)
    if match:
        return raw[match.start():]
    print("  WARNING: FLS anchor not found — using full text.")
    return raw


def remove_tables(text: str) -> str:
    text = RE_HEADER_ROW.sub(' ', text)
    text = RE_DATA_ROW.sub(' ', text)
    text = RE_FOLLOWING_TABLE.sub(' ', text)
    text = RE_ORPHAN_NUMS.sub(' ', text)
    text = RE_PAGE_NUM.sub(' ', text)
    return text


def clean(text: str) -> str:
    text = remove_tables(text)
    text = RE_ITEM_TAG.sub(' ', text)
    text = RE_PART_TAG.sub(' ', text)
    text = RE_COPYRIGHT.sub(' ', text)
    text = RE_WHITESPACE.sub(' ', text)
    return text.strip()


def get_sentences(text: str) -> list:
    return [s.strip() for s in sent_tokenize(text)
            if len(s.split()) >= MIN_SENT_TOKENS]


def normalise(sentence: str, remove_stops: bool = False) -> list:
    sentence = sentence.lower()
    sentence = re.sub(r"[^\w\s\-]", "", sentence)
    tokens   = sentence.split()
    tokens   = [t for t in tokens if not re.fullmatch(r'[\d,\.\-\%\$]+', t)]
    if remove_stops:
        tokens = [t for t in tokens if t not in STOPWORDS]
    return tokens

## N-gram Detection (gensim Phrases)
### gensim better than for loops for 20k doc, anf i think spacy is overkill for js text cleaning (and i dont hv gpu)


In [19]:
def build_phrase_models(all_token_lists):
    """
    Train bigram and trigram models on the full corpus.
    connector_words lets gensim bridge over 'of', 'the', 'and' etc.
    so 'results of operations' → 'results_of_operations'.
    """
    bigram = Phrases(
        all_token_lists,
        min_count=BIGRAM_MIN_COUNT,
        threshold=BIGRAM_THRESHOLD,
        connector_words=ENGLISH_CONNECTOR_WORDS,
    )
    trigram = Phrases(
        bigram[all_token_lists],
        min_count=BIGRAM_MIN_COUNT,
        threshold=BIGRAM_THRESHOLD,
        connector_words=ENGLISH_CONNECTOR_WORDS,
    )
    return bigram, trigram


def apply_phrases(tokens, bigram, trigram) -> list:
    """
    Apply bigram → trigram models, then blacklist filter.
    Hard cap: split any n-gram > 3 tokens back to unigrams.
    """
    phrased = list(trigram[bigram[tokens]])
    result = []
    for tok in phrased:
        parts = tok.split('_')
        if len(parts) <= 3 and not is_blacklisted(tok):
            result.append(tok)
        elif len(parts) > 3:
            result.extend(p for p in parts if not is_blacklisted(p))
        # else: blacklisted, drop it
    return result

## Main Pipeline
Reads all MDA files, cleans them, builds n-gram models on the full corpus, and produces two DataFrames.

In [20]:
def run_pipeline(mda_folder: str, file_pattern: str):
    """
    Returns:
        df_doc  — one row per filing (for company-level sentiment & topic modelling)
        df_sent — one row per sentence (for sentence-level sentiment × topic)
    """
    mda_path = Path(mda_folder)
    files = sorted(mda_path.glob(file_pattern))
    print(f"Found {len(files)} MDA files in {mda_folder}")

    if not files:
        raise FileNotFoundError(f"No files matching '{file_pattern}' in {mda_folder}")

    # ── Pass 1: read, parse metadata, clean, tokenise ──────────────
    print("Pass 1: reading and tokenising...")
    doc_store = {}  # keyed by filepath

    for fp in files:
        meta = parse_filename(fp.name)
        if meta is None:
            print(f"  SKIPPED (bad filename): {fp.name}")
            continue

        raw_text = fp.read_text(encoding='utf-8', errors='replace')
        text     = clean(strip_fls(raw_text))
        sents    = get_sentences(text)

        if not sents:
            print(f"  SKIPPED (no sentences after cleaning): {fp.name}")
            continue

        doc_id = f"{meta['company_name']}_{meta['filing_type']}_{meta['filing_date']}"

        doc_store[doc_id] = {
            "meta":       meta,
            "sentences":  sents,
            "raw_text":   text,
            "sent_tokens_clean": [normalise(s, remove_stops=True)  for s in sents],
            "sent_tokens_raw":   [normalise(s, remove_stops=False) for s in sents],
        }

    print(f"  {len(doc_store)} documents loaded successfully")

    # ── Train n-gram models on the FULL corpus ─────────────────────
    print("Training bigram/trigram models across full corpus...")
    all_clean_tokens = [tl for doc in doc_store.values()
                           for tl in doc["sent_tokens_clean"]]

    bigram, trigram = build_phrase_models(all_clean_tokens)
    print(f"  Phrase models trained on {len(all_clean_tokens)} sentences")

    # ── Pass 2: build both DataFrames ──────────────────────────────
    print("Pass 2: building output DataFrames...")
    doc_rows  = []
    sent_rows = []

    for doc_id, doc in doc_store.items():
        meta = doc["meta"]

        # Apply phrases to each sentence's clean tokens
        phrased_per_sent = [
            apply_phrases(tl, bigram, trigram)
            for tl in doc["sent_tokens_clean"]
        ]

        # ── Document-level row ─────────────────────────────────────
        all_clean_tokens_flat = [tok for sent_toks in phrased_per_sent
                                     for tok in sent_toks]

        doc_rows.append({
            "doc_id":       doc_id,
            "company_name": meta["company_name"],
            "filing_type":  meta["filing_type"],
            "filing_date":  meta["filing_date"],
            "year":         meta["year"],
            "quarter":      meta["quarter"],
            "raw_text":     doc["raw_text"],
            "sentences":    doc["sentences"],        # list[str]
            "clean_tokens": all_clean_tokens_flat,    # flat list[str]
            "n_sentences":  len(doc["sentences"]),
            "n_tokens":     len(all_clean_tokens_flat),
        })

        # ── Sentence-level rows ────────────────────────────────────
        for i, (sent, sent_toks) in enumerate(
                zip(doc["sentences"], phrased_per_sent)):
            sent_rows.append({
                "doc_id":       doc_id,
                "company_name": meta["company_name"],
                "filing_type":  meta["filing_type"],
                "filing_date":  meta["filing_date"],
                "year":         meta["year"],
                "quarter":      meta["quarter"],
                "sent_idx":     i,
                "sentence":     sent,               # original cleaned sentence
                "sent_tokens":  sent_toks,           # list[str] after stopwords + phrases
                "n_tokens":     len(sent_toks),
            })

    df_doc  = pd.DataFrame(doc_rows)
    df_sent = pd.DataFrame(sent_rows)

    # Sort for consistent output
    df_doc  = df_doc.sort_values(["company_name", "filing_date"]).reset_index(drop=True)
    df_sent = df_sent.sort_values(["company_name", "filing_date", "sent_idx"]).reset_index(drop=True)

    print(f"\nDone!")
    print(f"  df_doc:  {df_doc.shape[0]} documents × {df_doc.shape[1]} columns")
    print(f"  df_sent: {df_sent.shape[0]} sentences × {df_sent.shape[1]} columns")

    return df_doc, df_sent

## Run & Export

In [ ]:
df_doc, df_sent = run_pipeline(MDA_FOLDER, FILE_PATTERN)

# ── Quick inspection ───────────────────────────────────────────────
print("\n=== df_doc (first 3 rows, key columns) ===")
print(df_doc[["doc_id", "company_name", "filing_type", "filing_date",
              "n_sentences", "n_tokens"]].head(3).to_string(index=False))

print("\n=== df_sent (first 5 rows) ===")
print(df_sent[["doc_id", "sent_idx", "sentence", "n_tokens"]].head(5).to_string(index=False))

# ── Coverage summary ───────────────────────────────────────────────
print("\n=== Documents per company ===")
print(df_doc.groupby("company_name")["doc_id"].count().to_string())

Found 18183 MDA files in ../MDA
Pass 1: reading and tokenising...


In [ ]:
# ── Save to parquet (preserves list columns natively) ──────────────
df_doc.to_parquet("mda_doc_level.parquet",  index=False)
df_sent.to_parquet("mda_sent_level.parquet", index=False)
print("Saved: mda_doc_level.parquet, mda_sent_level.parquet")

# ── flat CSV for spot checking ───
df_sent_csv = df_sent.copy()
df_sent_csv["sent_tokens"] = df_sent_csv["sent_tokens"].apply(lambda x: ", ".join(x))
df_sent_csv.to_csv("mda_sent_level.csv", index=False)
print("Saved: mda_sent_level.csv (for inspection)")